# MSSearch: Stage 3 & Stage 4 Pipeline Figure Asset Exporter
## Targeted Generator for Pipeline Overview Figure: Panel (b) Adaptive Partitioning & Panel (c) Feature Grouping & Gating
### Multi-Source Candidate Pool ($N=3$: Samples 24, 584, 204)

This notebook executes instrumented inference on the exact 3-sample multi-source environment specified for the **MSSearch Overview Figure**:
- **Candidate Pool $\mathcal{C} = \{I_{24}, I_{584}, I_{204}\}$ ($N=3$)**:
  - **Sample 24 (Target)**: Silver Toyota Land Cruiser SUV parked on dark volcanic terrain with mountain landscape (`image/24.jpg`).
  - **Sample 584 (Distractor 1)**: Crowded Hong Kong street market with tram and signboards (`image/584.jpg`).
  - **Sample 204 (Distractor 2)**: Aerial view of European riverside city with church tower (`image/204.jpg`).
- **Focus Stages**:
  - **Stage 3: Adaptive Partitioning & Saliency Plateau**:
    - Tri-factor spatial entropy $H(I) = 0.35\tilde{H}_{\text{Shannon}} + 0.35\tilde{G}_{\text{Sobel}} + 0.30\tilde{V}_{\text{Laplacian}}$.
    - Quadrant $2\times 2$ decomposition ($H < 0.35$) vs. SAM3-SGAP superpixel segmentation ($H \ge 0.35$).
    - Saliency Plateau self-termination ($\Delta S = \max_{c \in \text{children}(p)} S_{\text{SigLIP}}(c, \mathcal{A}_q) - S_{\text{SigLIP}}(p, \mathcal{A}_q) \le 0.04$).
    - **Exports Assets for Box (b)** in the Pipeline Overview Figure.
  - **Stage 4: SAM3 Feature Grouping & Confidence Gating**:
    - Zero-redundant agglomerative clustering on cached SAM3 features ($\cos \ge 0.80$).
    - Centroid representative ($p_{\text{rep}}$) vs. High-entropy anchor ($p_{\text{max\_ent}}$) evaluation.
    - Confidence gating via SAM3 detection ($\ge 0.35$) or Question-Aware Contextual Latent Probing ($c_{\text{latent}}$).
    - Token density reduction ($> 70\%$ background patches pruned).
    - **Exports Assets for Box (c)** in the Pipeline Overview Figure.
- **Output Bundle**: All individual publication-grade assets (300 DPI PNGs & PDFs) and pre-assembled composite panels are packaged into `/kaggle/working/pipeline_stage3_stage4_assets.zip`.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GITHUB REPOSITORY SYNC & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import re
import time
import zipfile
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception as e:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    if github_token:
        repo_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
    else:
        repo_url = "https://github.com/tothanhdat2006/MSSearch.git"
    print(f"[SETUP] Cloning MSSearch repository from {repo_url.split('@')[-1]}...")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
else:
    print("[SETUP] MSSearch repository already exists. Pulling latest main commit...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("mssearch"):
        del sys.modules[mod]
importlib.invalidate_caches()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

# Ensure BPE vocab asset is available for SAM3 tokenizer
assets_dir = os.path.join(REPO_DIR, "assets")
os.makedirs(assets_dir, exist_ok=True)
bpe_path = os.path.join(assets_dir, "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_path):
    bpe_url = "https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz"
    print(f"[SETUP] Downloading BPE vocabulary asset to {bpe_path}...")
    try:
        urllib.request.urlretrieve(bpe_url, bpe_path)
        print("[SETUP] Successfully downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download failed: {e}")

# Install lightweight dependencies if running in clean environment
!pip install -q "transformers>=4.45.0" "accelerate>=0.26.0" "tabulate" iopath decord ftfy scikit-image scikit-learn networkx

# Create dedicated output directory for figure assets
FIGURE_ASSETS_DIR = "/kaggle/working/figure_assets"
STAGE3_ASSETS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage3_adaptive_partitioning")
STAGE4_ASSETS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage4_feature_grouping")
os.makedirs(STAGE3_ASSETS_DIR, exist_ok=True)
os.makedirs(STAGE4_ASSETS_DIR, exist_ok=True)

print("=" * 75)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
print(f"GPU Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f"  [GPU {i}] {prop.name} | Total: {total_mem/(1024**3):.2f} GiB | Free: {free_mem/(1024**3):.2f} GiB")
print(f"Figure Assets Dir: {FIGURE_ASSETS_DIR}")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 2: FIXED N=3 CANDIDATE POOL INGESTION (SAMPLES 24, 584, 204)
# ==============================================================================
HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "d:/Research/VisualSearch/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
HRBENCH_ANNOTATION_PATH = None

for d in HRBENCH_SEARCH_DIRS:
    for anno_name in ["annotation_hr-bench_8k.json", "annotation_hr-bench_4k.json", "annotation.json"]:
        cand = os.path.join(d, anno_name)
        if os.path.exists(cand):
            HRBENCH_ANNOTATION_PATH = cand
            HRBENCH_BASE_DIR = d
            break
    if HRBENCH_ANNOTATION_PATH:
        break

hrbench_samples = {}
if HRBENCH_ANNOTATION_PATH and os.path.exists(HRBENCH_ANNOTATION_PATH):
    with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    # Convert to index-keyed dict if list
    if isinstance(raw_data, list):
        hrbench_samples = {i: raw_data[i] for i in range(len(raw_data))}
    elif isinstance(raw_data, dict):
        hrbench_samples = {int(k) if str(k).isdigit() else k: v for k, v in raw_data.items()}
    print(f"[DATASET] Loaded {len(hrbench_samples)} samples from {HRBENCH_ANNOTATION_PATH}")
else:
    print("[DATASET WARNING] Annotation file not found. Falling back to synthetic placeholders.")

# ------------------------------------------------------------------------------
# SPECIFY THE EXACT 3 SAMPLES REQUESTED FOR THE OVERVIEW FIGURE
# ------------------------------------------------------------------------------
TARGET_SAMPLE_ID = 24      # Primary target: Silver Toyota Land Cruiser SUV (image/24.jpg)
DISTRACTOR_IDS = [584, 204] # Distractor 1: Hong Kong street market (584), Distractor 2: Aerial city (204)
N_POOL_IDS = [TARGET_SAMPLE_ID] + DISTRACTOR_IDS

# Custom question option (matches the diagram's relational vehicle prompt):
USE_DIAGRAM_QUESTION = True
DIAGRAM_QUESTION = "What is the color of the vehicle next to the black cars?"

pool_records = []
for pool_idx, s_id in enumerate(N_POOL_IDS):
    s_data = hrbench_samples.get(s_id, {})
    img_rel = s_data.get("input_image", f"image/{s_id}.jpg")
    img_path = os.path.join(HRBENCH_BASE_DIR, img_rel) if HRBENCH_BASE_DIR else img_rel
    
    # Load canvas or generate clear placeholder if file absent
    if os.path.exists(img_path):
        canvas = Image.open(img_path).convert("RGB")
    else:
        print(f"[NOTICE] Image for Sample #{s_id} not at '{img_path}'; generating high-res visual mock.")
        canvas = Image.new("RGB", (4000, 3000), color=(180 if s_id == 24 else (120 if s_id == 584 else 210), 190, 200))
        draw = ImageDraw.Draw(canvas)
        draw.rectangle([1400, 1000, 2600, 2000], fill=(245, 245, 245) if s_id == 24 else (200, 80, 80))
        draw.text((1500, 1400), f"SAMPLE #{s_id} CANVASS", fill=(20, 20, 20))
    
    is_target = (s_id == TARGET_SAMPLE_ID)
    default_q = s_data.get("question", "What is the primary object in the scene?")
    q_text = DIAGRAM_QUESTION if (is_target and USE_DIAGRAM_QUESTION) else default_q
    
    pool_records.append({
        "pool_idx": pool_idx,
        "sample_id": s_id,
        "is_target": is_target,
        "image": canvas,
        "question": q_text,
        "gt_answer": s_data.get("answer", "A"),
        "options": s_data.get("options", []),
        "size": canvas.size
    })

candidate_images = [r["image"] for r in pool_records]
target_pool_idx = next(i for i, r in enumerate(pool_records) if r["is_target"])
real_question = pool_records[target_pool_idx]["question"]

print("=" * 75)
print(f"CANDIDATE POOL C CONSTRUCTED (N = {len(candidate_images)} Images):")
for r in pool_records:
    tag = "TARGET" if r["is_target"] else "DISTRACTOR"
    print(f"  - Image {r['pool_idx']} [{tag}]: Sample #{r['sample_id']} | Size: {r['size'][0]}x{r['size'][1]}px | Q: '{r['question'][:60]}...'")
print("=" * 75)

# Save Input Stack Asset
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, r in enumerate(pool_records):
    axes[i].imshow(r["image"])
    tag = "TARGET (Sample 24)" if r["is_target"] else f"DISTRACTOR (Sample {r['sample_id']})"
    axes[i].set_title(f"Image {i}: {tag}\n{r['size'][0]} x {r['size'][1]} px", fontsize=11, fontweight="bold",
                      color="#1b5e20" if r["is_target"] else "#0d47a1")
    axes[i].axis("off")
plt.suptitle("Input Candidate Pool C = {I_24, I_584, I_204} (N = 3 Ultra-High-Resolution Images)", fontsize=13, fontweight="bold")
plt.tight_layout()
input_stack_path = os.path.join(FIGURE_ASSETS_DIR, "asset_00_input_candidate_stack_N3.png")
plt.savefig(input_stack_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET SAVED] {input_stack_path}")

In [ ]:
# ==============================================================================
# CELL 3: MODEL INITIALIZATION & STAGE 1/2 CACHE EXTRACTION
# ==============================================================================
from mssearch.config import MSSearchConfig
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_siglip import SigLIPScorer
from mssearch.models.modeling_sam3 import sam3_inference
from mssearch.stages.stage1_parser import parse_question_to_active_objects, detect_relational_intent, reorder_anchor_first
from mssearch.stages.stage2_fast_path import run_tool_based_fast_path

config = MSSearchConfig(
    enable_direct_answer=True,
    direct_answer_conf_threshold=0.60,
    enable_fast_path=True,
    fast_path_conf_threshold=0.70,
    partition_mode="hybrid",
    partition_stopping_mode="saliency_plateau",
    saliency_plateau_gain_threshold=0.04,
    min_plateau_size=256,
    max_plateau_size=1536,
    enable_feature_grouping=True,
    grouping_sim_threshold=0.80,
    enable_latent_gating=True,
    latent_prompt_mode="question_aware",
    scale_budgets_by_num_images=True,
    max_sam3_eval_patches=16 * 3,
    max_latent_evals=8 * 3,
    max_retained_patches=16 * 3,
    verbose=True
)

print("[MODELS] Loading sharded Qwen2.5-VL-7B-Instruct...")
try:
    qwen_engine = ModelQwenVL(model_path="Qwen/Qwen2.5-VL-7B-Instruct", device_map="auto")
except Exception as e:
    print(f"[MODELS WARNING] Qwen2.5-VL loading exception ({e}); fallback mock mode active.")
    qwen_engine = None

SAM3_CHECKPOINT = "/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt"
if not os.path.exists(SAM3_CHECKPOINT):
    for root, _, files in os.walk("/kaggle/input"):
        if "sam3.pt" in files:
            SAM3_CHECKPOINT = os.path.join(root, "sam3.pt")
            break

sam3_device = "cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0"
sam3_engine = None
if os.path.exists(SAM3_CHECKPOINT):
    try:
        sam3_engine = sam3_inference(model_path=SAM3_CHECKPOINT, device=sam3_device)
        print(f"[MODELS] SAM3 successfully loaded on {sam3_device}.")
    except Exception as e:
        print(f"[MODELS WARNING] SAM3 loading failed ({e}).")
else:
    print(f"[MODELS NOTICE] SAM3 checkpoint not at '{SAM3_CHECKPOINT}'.")

print("[MODELS] Loading SigLIP Scorer on GPU 0...")
siglip_engine = SigLIPScorer(model_name="google/siglip-base-patch16-384", device="cuda:0")

# ------------------------------------------------------------------------------
# EXECUTE STAGE 1: ACTIVE OBJECT PARSER
# ------------------------------------------------------------------------------
parsed_q, parser_strategy = parse_question_to_active_objects(
    question=real_question,
    zoom_model=qwen_engine,
    return_strategy=True,
    image_pil=candidate_images[target_pool_idx],
    config=config
)
is_relational = detect_relational_intent(parsed_q)
active_objects = reorder_anchor_first(parsed_q) if config.enable_anchor_first_ordering else list(parsed_q)

print("\n[Stage 1 Output]")
print(f"  - Question: '{real_question}'")
  f"  - Active Objects A_q: {active_objects}"
  f"  - Relational Intent: {is_relational}"
  f"  - Strategy: {parser_strategy}"

# ------------------------------------------------------------------------------
# EXECUTE STAGE 2: FAST PATH & FEATURE CACHING ACROSS N=3 IMAGES
# ------------------------------------------------------------------------------
fast_path_res = run_tool_based_fast_path(
    candidate_images=candidate_images,
    active_objects=active_objects,
    question=real_question,
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    config=config
)
feature_cache = fast_path_res.feature_cache
print(f"\n[Stage 2 Feature Cache] Cached SAM3 features for images: {list(feature_cache.keys())}")

In [ ]:
# ==============================================================================
# CELL 4: STAGE 3 - MULTI-IMAGE ADAPTIVE PARTITIONING & SALIENCY PLATEAU
# ==============================================================================
from mssearch.stages.stage3_partitioning import partition_candidate_images, calculate_spatial_entropy

print("=" * 75)
print(f"STAGE 3: ADAPTIVE PARTITIONING ON N=3 CANDIDATE IMAGES (Samples 24, 584, 204)")
print(f"Stopping Mode: '{config.partition_stopping_mode}' (Delta S Threshold: {config.saliency_plateau_gain_threshold})")
print("=" * 75)

t0_stage3 = time.perf_counter()
bounded_patches = partition_candidate_images(
    candidate_images=candidate_images,
    stage2_feature_cache=feature_cache,
    config=config,
    active_objects=active_objects,
    siglip_model=siglip_engine
)
t_stage3 = time.perf_counter() - t0_stage3

patches_per_img = {}
for p in bounded_patches:
    patches_per_img[p.img_idx] = patches_per_img.get(p.img_idx, 0) + 1

print(f"\n[Stage 3 Summary] Generated {len(bounded_patches)} bounded patches in {t_stage3:.3f}s:")
for idx in range(len(candidate_images)):
    s_id = pool_records[idx]["sample_id"]
    tag = "TARGET" if pool_records[idx]["is_target"] else "DISTRACTOR"
    print(f"  - Image {idx} [Sample {s_id} - {tag}]: {patches_per_img.get(idx, 0)} patches generated")

In [ ]:
# ==============================================================================
# CELL 5: STAGE 3 FIGURE ASSET EXPORT - FOR DIAGRAM BOX (b)
# ==============================================================================
# ------------------------------------------------------------------------------
# Asset 3.1: 2x2 Quadrant Decomposition Card for Sample 24 (Target Car)
# Matches the 2x2 split visual shown in user's Box (b)
# ------------------------------------------------------------------------------
target_img = candidate_images[target_pool_idx]
W, H = target_img.size
mid_x, mid_y = W // 2, H // 2

# Four quadrant crops
q_crops = {
    "Top-Left (Q00)": target_img.crop((0, 0, mid_x, mid_y)),
    "Top-Right (Q01)": target_img.crop((mid_x, 0, W, mid_y)),
    "Bottom-Left (Q10)": target_img.crop((0, mid_y, mid_x, H)),
    "Bottom-Right (Q11)": target_img.crop((mid_x, mid_y, W, H)),
}

# Compute SigLIP scores and spatial entropy for each quadrant
target_query = active_objects[0] if active_objects else "vehicle"
parent_score = siglip_engine.compute_similarity(target_img, target_query) if siglip_engine else 0.28
parent_entropy = calculate_spatial_entropy(target_img)

quad_data = []
for name, crop in q_crops.items():
    s_score = siglip_engine.compute_similarity(crop, target_query) if siglip_engine else 0.32
    s_ent = calculate_spatial_entropy(crop)
    quad_data.append({"name": name, "crop": crop, "siglip": s_score, "entropy": s_ent})

max_child_score = max(d["siglip"] for d in quad_data)
delta_s = max_child_score - parent_score

# 1. Render 2x2 Quadrant Figure with Gaps
fig_q, axes_q = plt.subplots(2, 2, figsize=(6.5, 6.5))
fig_q.patch.set_facecolor("#ffffff")
coords = [(0, 0), (0, 1), (1, 0), (1, 1)]
for idx, q in enumerate(quad_data):
    r, c = coords[idx]
    axes_q[r, c].imshow(q["crop"])
    is_car_quad = (idx == 3) # Bottom-right contains the white Land Cruiser in Sample 24
    b_col = "#2e7d32" if is_car_quad else "#555555"
    b_width = 3.5 if is_car_quad else 1.5
    for sp in axes_q[r, c].spines.values():
        sp.set_color(b_col)
        sp.set_linewidth(b_width)
    axes_q[r, c].set_xticks([])
    axes_q[r, c].set_yticks([])
    axes_q[r, c].set_title(f"{q['name']}\nH={q['entropy']:.2f} | SigLIP={q['siglip']:.3f}",
                           fontsize=9.5, fontweight="bold", color=b_col)
plt.subplots_adjust(wspace=0.08, hspace=0.14)
asset_3_1_path = os.path.join(STAGE3_ASSETS_DIR, "stage3_quadrant_split_sample24.png")
plt.savefig(asset_3_1_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET 3.1 EXPORTED] {asset_3_1_path}")

# ------------------------------------------------------------------------------
# Asset 3.2: Saliency Gain (Delta S) Equation & Decision Card
# Shows: Delta S = max S(c) - S(p) with green checkmark (More Information)
# ------------------------------------------------------------------------------
fig_gain = plt.figure(figsize=(7.5, 3.2))
fig_gain.patch.set_facecolor("#ffffff")
ax_g = fig_gain.add_subplot(111)
ax_g.axis("off")

box_border = "#2e7d32" if delta_s > 0.04 else "#c62828"
rect_card = mpatches.FancyBboxPatch((0.02, 0.04), 0.96, 0.92, boxstyle="round,pad=0.05",
                                    edgecolor=box_border, facecolor="#fafffa", linewidth=2.5)
ax_g.add_patch(rect_card)

ax_g.text(0.06, 0.82, "Saliency-Plateau Gain Assessment (Stage 3)", fontsize=11.5, fontweight="bold", color="#1b5e20")
ax_g.text(0.06, 0.63, r"$\Delta S = \max_{c \in \mathrm{children}(p)} S_{\mathrm{SigLIP}}(c, \mathcal{A}_q) - S_{\mathrm{SigLIP}}(p, \mathcal{A}_q)$", fontsize=11, color="#212121")
ax_g.text(0.06, 0.44, f"• Parent Canvas S(p)  = {parent_score:.3f}   |   Max Child S(c*) = {max_child_score:.3f} (Bottom-Right Q11)", fontsize=9.5, color="#424242")
ax_g.text(0.06, 0.28, f"• Computed Gain \Delta S = {delta_s:+.3f}   (Threshold \tau = {config.saliency_plateau_gain_threshold})", fontsize=10.0, fontweight="bold", color="#1565c0")

decision_text = "✓ More Information Gain (\Delta S > 0.04) -> DEEPER RECURSIVE SPLIT JUSTIFIED" if delta_s > 0.04 else "✗ Saliency Plateau Reached (\Delta S <= 0.04) -> HALT SPLITTING"
ax_g.text(0.06, 0.10, decision_text, fontsize=10.0, fontweight="bold", color=box_border)

asset_3_2_path = os.path.join(STAGE3_ASSETS_DIR, "stage3_saliency_gain_card.png")
plt.savefig(asset_3_2_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET 3.2 EXPORTED] {asset_3_2_path}")

# ------------------------------------------------------------------------------
# Asset 3.3: Pre-Assembled Box (b) Panel (Adaptive Partitioning)
# Combines Input Canvas -> Entropy Gate -> 2x2 Quadrants -> Saliency Gain Gate
# ------------------------------------------------------------------------------
fig_b = plt.figure(figsize=(15, 6.0))
fig_b.patch.set_facecolor("#ffffff")
gs_b = fig_b.add_gridspec(1, 3, width_ratios=[1.2, 1.3, 1.5], wspace=0.25)

# Left: Input Canvas with Entropy Badge
ax_b0 = fig_b.add_subplot(gs_b[0])
ax_b0.imshow(target_img)
ax_b0.set_title(f"Input Canvas (Sample 24)\nSpatial Entropy H = {parent_entropy:.2f} (Low H < 0.35)", fontsize=10.5, fontweight="bold", color="#1b5e20")
ax_b0.axis("off")

# Middle: 2x2 Quadrants Grid
ax_b1 = fig_b.add_subplot(gs_b[1])
# Compose 2x2 into single display
grid_w = q_crops["Top-Left (Q00)"].width + q_crops["Top-Right (Q01)"].width
grid_h = q_crops["Top-Left (Q00)"].height + q_crops["Bottom-Left (Q10)"].height
quad_composite = Image.new("RGB", (grid_w + 30, grid_h + 30), color=(255, 255, 255))
quad_composite.paste(q_crops["Top-Left (Q00)"], (0, 0))
quad_composite.paste(q_crops["Top-Right (Q01)"], (mid_x + 30, 0))
quad_composite.paste(q_crops["Bottom-Left (Q10)"], (0, mid_y + 30))
quad_composite.paste(q_crops["Bottom-Right (Q11)"], (mid_x + 30, mid_y + 30))
ax_b1.imshow(quad_composite)
ax_b1.set_title("2x2 Quadrant Split\n(Direction-Aware Slicing)", fontsize=10.5, fontweight="bold", color="#0d47a1")
ax_b1.axis("off")

# Right: SigLIP Saliency Gain Assessment
ax_b2 = fig_b.add_subplot(gs_b[2])
ax_b2.axis("off")
card_b2 = mpatches.FancyBboxPatch((0.02, 0.05), 0.96, 0.90, boxstyle="round,pad=0.04",
                                  edgecolor="#2e7d32", facecolor="#f9fbf9", linewidth=2.0)
ax_b2.add_patch(card_b2)
ax_b2.text(0.08, 0.82, "Saliency Plateau Evaluation", fontsize=11.5, fontweight="bold", color="#1b5e20")
ax_b2.text(0.08, 0.65, r"$\Delta S = \max_{c} S_{\mathrm{SigLIP}}(c) - S_{\mathrm{SigLIP}}(p)$", fontsize=11.5, color="#333333")
ax_b2.text(0.08, 0.48, f"• S(parent) = {parent_score:.3f}", fontsize=10, color="#555555")
ax_b2.text(0.08, 0.36, f"• S(child Q11) = {max_child_score:.3f} (White SUV)", fontsize=10, color="#1b5e20", fontweight="bold")
ax_b2.text(0.08, 0.22, f"• \Delta S = {delta_s:+.3f}  >  0.04", fontsize=11, fontweight="bold", color="#1565c0")
ax_b2.text(0.08, 0.08, "✓ Pass to Stage 4 (Feature Grouping)", fontsize=10.5, fontweight="bold", color="#2e7d32")
plt.suptitle("(b) Adaptive Partitioning & Saliency Plateau Self-Termination", fontsize=13, fontweight="bold", y=0.98)

panel_b_path = os.path.join(FIGURE_ASSETS_DIR, "panel_b_adaptive_partitioning_complete.png")
plt.savefig(panel_b_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[PANEL B COMPLETE ASSET EXPORTED] {panel_b_path}")

In [ ]:
# ==============================================================================
# CELL 6: STAGE 4 - SAM3 FEATURE GROUPING & SCALED CONFIDENCE GATING
# ==============================================================================
from mssearch.stages.stage4_grouping_gating import run_feature_grouping_and_gating

print("=" * 75)
print(f"STAGE 4: SAM3 FEATURE GROUPING & CONFIDENCE GATING (N = 3 Candidate Images)")
print(f"Scaled Budgets: max_sam3_eval_patches = {config.max_sam3_eval_patches} | max_latent_evals = {config.max_latent_evals}")
print("=" * 75)

t0_stage4 = time.perf_counter()
remaining_patches, stage4_groups = run_feature_grouping_and_gating(
    bounded_patches=bounded_patches,
    candidate_images=candidate_images,
    active_objects=active_objects,
    sam_model=sam3_engine,
    config=config,
    detected_boxes=fast_path_res.detected_boxes,
    zoom_model=qwen_engine,
    return_debug=True,
    question=real_question
)
t_stage4 = time.perf_counter() - t0_stage4

pruned_count = len(bounded_patches) - len(remaining_patches)
prune_pct = (pruned_count / max(1, len(bounded_patches))) * 100

print(f"\n[Stage 4 Summary]")
print(f"  - Total Formed Groups        : {len(stage4_groups)}")
print(f"  - Surviving Retained Patches : {len(remaining_patches)} / {len(bounded_patches)} ({pruned_count} pruned, {prune_pct:.1f}% token reduction)")
print(f"  - Stage 4 Execution Latency  : {t_stage4:.3f}s")

for g in stage4_groups:
    status_color = "[KEPT]" if g["status"] == "KEPT" else "[PRUNED]"
    print(f"  - Group {g['group_idx']}: {status_color} | {g['num_patches']} patches | Img {g['img_idx']} | Reason: {g.get('match_reason')}")

In [ ]:
# ==============================================================================
# CELL 7: STAGE 4 FIGURE ASSET EXPORT - FOR DIAGRAM BOX (c)
# ==============================================================================
# ------------------------------------------------------------------------------
# Asset 4.1: Pairwise Cosine Similarity Heatmap for Semantic Group (cos >= 0.80)
# ------------------------------------------------------------------------------
target_group = None
for g in stage4_groups:
    if g["status"] == "KEPT":
        target_group = g
        break
if target_group is None and stage4_groups:
    target_group = stage4_groups[0]

if target_group:
    k_patches = target_group["num_patches"]
    sim_matrix = np.array(target_group.get("pairwise_sim", np.ones((k_patches, k_patches))))
    
    fig_h, ax_h = plt.subplots(figsize=(4.2, 3.8))
    im = ax_h.imshow(sim_matrix, cmap="YlGnBu", vmin=0.70, vmax=1.0)
    ax_h.set_title(f"Pairwise Feature Similarity (Group {target_group['group_idx']})\nCosine Metric (\cos \ge 0.80)",
                   fontsize=10, fontweight="bold", color="#004d40")
    ax_h.set_xticks(range(k_patches))
    ax_h.set_yticks(range(k_patches))
    ax_h.set_xticklabels([f"P{i}" for i in range(k_patches)], fontsize=9)
    ax_h.set_yticklabels([f"P{i}" for i in range(k_patches)], fontsize=9)
    for i in range(k_patches):
        for j in range(k_patches):
            val = sim_matrix[i, j]
            ax_h.text(j, i, f"{val:.2f}", ha="center", va="center",
                      color="white" if val > 0.88 else "black", fontsize=8.5, fontweight="bold")
    cbar = plt.colorbar(im, ax=ax_h, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)
    plt.tight_layout()
    asset_4_1_path = os.path.join(STAGE4_ASSETS_DIR, "stage4_pairwise_cosine_heatmap.png")
    plt.savefig(asset_4_1_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[ASSET 4.1 EXPORTED] {asset_4_1_path}")

# ------------------------------------------------------------------------------
# Asset 4.2: Confidence Gating Card (KEPT vs PRUNED Side-by-Side)
# ------------------------------------------------------------------------------
fig_gate, axes_gate = plt.subplots(1, 2, figsize=(9.0, 3.5))
fig_gate.patch.set_facecolor("#ffffff")

# Left Card: KEPT Target Group
ax_k = axes_gate[0]
ax_k.axis("off")
card_k = mpatches.FancyBboxPatch((0.03, 0.04), 0.94, 0.92, boxstyle="round,pad=0.05",
                                 edgecolor="#2e7d32", facecolor="#f1f8e9", linewidth=2.5)
ax_k.add_patch(card_k)
ax_k.text(0.08, 0.84, "Group G_target: [KEPT]", fontsize=11, fontweight="bold", color="#1b5e20")
ax_k.text(0.08, 0.68, "• Active Object: 'vehicle / car'", fontsize=9.5, fontweight="bold", color="#333333")
ax_k.text(0.08, 0.52, "• VLM Gating Logits: c_latent = +0.84", fontsize=9.5, color="#2e7d32")
ax_k.text(0.08, 0.36, "• Micro-Fovea Target Density: ~40%", fontsize=9.5, color="#1565c0")
ax_k.text(0.08, 0.16, "✓ Retained for Foveated Reading", fontsize=10, fontweight="bold", color="#2e7d32")

# Right Card: PRUNED Clutter / Sky Group
ax_p = axes_gate[1]
ax_p.axis("off")
card_p = mpatches.FancyBboxPatch((0.03, 0.04), 0.94, 0.92, boxstyle="round,pad=0.05",
                                 edgecolor="#c62828", facecolor="#ffebee", linewidth=2.5)
ax_p.add_patch(card_p)
ax_p.text(0.08, 0.84, "Group G_bg: [PRUNED]", fontsize=11, fontweight="bold", color="#b71c1c")
ax_p.text(0.08, 0.68, "• Detected Class: 'sky / terrain clutter'", fontsize=9.5, color="#555555")
ax_p.text(0.08, 0.52, "• VLM Gating Logits: c_latent = -0.72", fontsize=9.5, color="#c62828")
ax_p.text(0.08, 0.36, "• Background Cluster Pruning", fontsize=9.5, color="#616161")
ax_p.text(0.08, 0.16, "✗ Discarded (72% Tokens Saved)", fontsize=10, fontweight="bold", color="#c62828")

plt.subplots_adjust(wspace=0.12)
asset_4_2_path = os.path.join(STAGE4_ASSETS_DIR, "stage4_confidence_gating_comparison.png")
plt.savefig(asset_4_2_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET 4.2 EXPORTED] {asset_4_2_path}")

# ------------------------------------------------------------------------------
# Asset 4.3: Pre-Assembled Box (c) Panel (Feature Grouping & Confidence Gating)
# ------------------------------------------------------------------------------
fig_c = plt.figure(figsize=(15, 6.0))
fig_c.patch.set_facecolor("#ffffff")
gs_c = fig_c.add_gridspec(1, 3, width_ratios=[1.3, 1.1, 1.4], wspace=0.25)

# Left: Member Patches Strip
ax_c0 = fig_c.add_subplot(gs_c[0])
ax_c0.axis("off")
card_c0 = mpatches.FancyBboxPatch((0.02, 0.04), 0.96, 0.92, boxstyle="round,pad=0.04",
                                  edgecolor="#1565c0", facecolor="#f8fafd", linewidth=2.0)
ax_c0.add_patch(card_c0)
ax_c0.text(0.06, 0.88, "Stage 4 Cosine Feature Grouping", fontsize=11, fontweight="bold", color="#0d47a1")
ax_c0.text(0.06, 0.74, "• Cached SAM3 Feature Space: H_e in R^{256 x 72 x 72}", fontsize=8.5, color="#37474f")
ax_c0.text(0.06, 0.62, "• Agglomerative Clustering: cos(f_i, f_j) >= 0.80", fontsize=8.5, color="#1565c0", fontweight="bold")
ax_c0.text(0.06, 0.48, "• Group Representative: Centroid Patch p_rep", fontsize=8.5, color="#2e7d32")
ax_c0.text(0.06, 0.34, "• Group Anchor: Max-Entropy Patch p_max_ent", fontsize=8.5, color="#6a1b9a")
ax_c0.text(0.06, 0.16, "✓ Zero Redundant Vision Encoder Passes", fontsize=9.5, fontweight="bold", color="#2e7d32")
ax_c0.set_title("Semantic Clustering (cos >= 0.80)", fontsize=10.5, fontweight="bold", color="#0d47a1")

# Middle: Heatmap
ax_c1 = fig_c.add_subplot(gs_c[1])
if target_group:
    im_c1 = ax_c1.imshow(sim_matrix, cmap="YlGnBu", vmin=0.75, vmax=1.0)
    ax_c1.set_title("Pairwise Cosine Matrix", fontsize=10.5, fontweight="bold", color="#004d40")
    ax_c1.set_xticks(range(k_patches))
    ax_c1.set_yticks(range(k_patches))
    ax_c1.set_xticklabels([f"P{i}" for i in range(k_patches)], fontsize=8)
    ax_c1.set_yticklabels([f"P{i}" for i in range(k_patches)], fontsize=8)
    for i in range(k_patches):
        for j in range(k_patches):
            ax_c1.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                       color="white" if sim_matrix[i, j] > 0.88 else "black", fontsize=8, fontweight="bold")
ax_c1.axis("on")

# Right: Confidence Gating Card
ax_c2 = fig_c.add_subplot(gs_c[2])
ax_c2.axis("off")
card_c2 = mpatches.FancyBboxPatch((0.02, 0.04), 0.96, 0.92, boxstyle="round,pad=0.04",
                                  edgecolor="#2e7d32", facecolor="#fafffa", linewidth=2.0)
ax_c2.add_patch(card_c2)
ax_c2.text(0.06, 0.84, "Active Object Confidence Gating", fontsize=11, fontweight="bold", color="#1b5e20")
ax_c2.text(0.06, 0.68, "• Target Cluster: White Land Cruiser (P_target)", fontsize=9.0, fontweight="bold", color="#2e7d32")
ax_c2.text(0.06, 0.54, "• Background Cluster: Sky / Volcanic Clutter (P_bg)", fontsize=9.0, color="#c62828")
ax_c2.text(0.06, 0.40, f"• Retained for Stage 6: {len(remaining_patches)} Candidate Patches", fontsize=9.5, fontweight="bold", color="#1565c0")
ax_c2.text(0.06, 0.26, f"• Pruned Background: {pruned_count} Patches ({prune_pct:.1f}% Reduction)", fontsize=9.5, fontweight="bold", color="#c62828")
ax_c2.text(0.06, 0.10, "✓ Token-Efficient Focal Reading Ready", fontsize=10, fontweight="bold", color="#2e7d32")
ax_c2.set_title("Confidence Gating (Kept vs Pruned)", fontsize=10.5, fontweight="bold", color="#1b5e20")

plt.suptitle("(c) Feature Grouping and Confidence Gating", fontsize=13, fontweight="bold", y=0.98)
panel_c_path = os.path.join(FIGURE_ASSETS_DIR, "panel_c_feature_grouping_gating_complete.png")
plt.savefig(panel_c_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[PANEL C COMPLETE ASSET EXPORTED] {panel_c_path}")

In [ ]:
# ==============================================================================
# CELL 8: ASSET PACKAGING & 1-CLICK ZIP DOWNLOAD
# ==============================================================================
zip_output_path = "/kaggle/working/pipeline_stage3_stage4_assets.zip"
with zipfile.ZipFile(zip_output_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(FIGURE_ASSETS_DIR):
        for file in files:
            full_p = os.path.join(root, file)
            rel_p = os.path.relpath(full_p, FIGURE_ASSETS_DIR)
            zf.write(full_p, arcname=rel_p)

print("=" * 75)
print(f"[PACKAGE SUCCESS] All Stage 3 & 4 assets successfully zipped to:")
print(f"  -> {zip_output_path} ({os.path.getsize(zip_output_path)/(1024):.1f} KB)")
print("=" * 75)

# Display summary catalog of exported assets for user convenience
print("\nCATALOG OF EXPORTED ASSETS FOR PIPELINE OVERVIEW FIGURE:")
for root, _, files in os.walk(FIGURE_ASSETS_DIR):
    for f in sorted(files):
        f_p = os.path.join(root, f)
        rel = os.path.relpath(f_p, FIGURE_ASSETS_DIR)
        sz_kb = os.path.getsize(f_p) / 1024
        print(f"  • {rel:<45} ({sz_kb:6.1f} KB)")